# JN0h · The agent instruction file

*On-ramp 8 of 8 — the last before JN1.*

How do you make an agent follow *your* project's rules instead of generic defaults that might quietly corrupt your data? You write them down — in an **instruction file** the agent reads first.

### Running the cells

To run a cell, click it and press **Shift + Return**, or click the **run (▸) button** on the cell. The simplest way through any notebook here is to start at the top and run each cell in order, reading the output that appears beneath it.

Some of the computational cells may look complex right now — that's expected, and it's fine. **You don't need to understand every line yet;** the ideas become clear as you go. Run them, watch what they produce, and keep moving.

## (run first) Colab setup

Fetches the data + shared modules from R2. **No-op if you already have the repo locally.** On Colab it recreates the minimal layout.

In [1]:
# === COLAB BOOTSTRAP - fetch curriculum data + modules from R2 (NO-OP if the repo is local) ===
from pathlib import Path
import sys, urllib.request, urllib.parse, tarfile, subprocess

R2 = 'https://pub-2cee87f70da64080ab70ee0a34b55099.r2.dev/curriculum'
USE_CLEAN = False   # False: raw .xlsx path (JN1's messy-data lesson).  True (skip-ingest): permits_clean.*

_here = Path.cwd()
_have_repo = (_here/'scripts'/'build_v2').exists() or any((p/'scripts'/'build_v2').exists() for p in _here.parents)

def _get(url):
    # r2.dev sits behind Cloudflare, which 403s the default 'Python-urllib' User-Agent; send a browser UA.
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=60) as r:
        return r.read()

if _have_repo:
    print('local repo detected - no fetch needed')
else:
    try:
        import pyarrow  # the parquet / USE_CLEAN path needs it; Colab has pandas, maybe not pyarrow
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow'], check=True)
    def _fetch(url, dest):
        dest = Path(dest)
        if dest.exists():
            return                                   # cached: re-runs don't re-download
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(_get(url)); print('fetched', dest.name)
    # 1) shared modules -> ./scripts/...  (the config-cell repo-root walk then finds scripts/build_v2)
    if not (_here/'scripts'/'build_v2').exists():
        Path('modules.tgz').write_bytes(_get(f'{R2}/curriculum_modules.tar.gz'))
        _tar = tarfile.open('modules.tgz')
        try: _tar.extractall(_here, filter='data')      # py3.12+: safe extract, no deprecation warning
        except TypeError: _tar.extractall(_here)         # older python has no filter arg
        _tar.close(); Path('modules.tgz').unlink(missing_ok=True)   # tidy: drop the intermediate tarball
        print('extracted modules -> ./scripts/')
    # 2) data -> the SAME relative paths the notebooks use (raw .xlsx AND clean exports, both fetched)
    for rel in ['data/raw/cpra-downloads/BP_Annual Permit Report-2018-2022.xlsx',
                'data/raw/cpra-downloads/BP_Annual Permit Report-2023-2025.xlsx',
                'databases/hcd_apr_mirror_2026-06-17_fresh.db',
                'databases/hcd_apr_mirror.db',
                'data/processed/permits_clean.csv',
                'data/processed/permits_clean.parquet',
                'data/processed/permits_clean_README.md']:
        _fetch(f"{R2}/data/{urllib.parse.quote(rel.split('/')[-1])}", _here/rel)   # quote -> %20 for the spaced .xlsx names
    print('curriculum bundle ready (fetched from R2)')


local repo detected - no fetch needed


In [2]:
def md(t):
    from IPython.display import Markdown, display
    display(Markdown(t))

## Point the notebook at the data

Finds the repo root, the permit feed, and the project's shared code. The two knobs near the top are all a student changes to run another city.

In [3]:
# === CONFIG — point this at YOUR city's permit data (this notebook is clonable) ===
from pathlib import Path
import sys, glob

# walk up to the repo root (where scripts/build_v2 lives) so the notebook runs from anywhere
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'scripts' / 'build_v2').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

# --- the two knobs a student changes for another city ---
PERMIT_GLOB   = str(REPO_ROOT / 'data/raw/cpra-downloads/BP_Annual Permit Report-*.xlsx')
HEADER_ROW    = 7        # 0-indexed: Berkeley's CPRA export puts the column names on row 8
EXPECTED_UNIQUE = 30764  # the known unique-permit total for YOUR feed (Berkeley = 30,764)

# import the REAL shared modules the pipeline uses (we demonstrate them, never reinvent)
sys.path.insert(0, str(REPO_ROOT / 'scripts'))
sys.path.insert(0, str(REPO_ROOT / 'scripts' / 'build_v2'))
print('repo root :', REPO_ROOT)
print('feed files:', [Path(f).name for f in glob.glob(PERMIT_GLOB)])


repo root : /Users/johngage/berkeley-data
feed files: ['BP_Annual Permit Report-2023-2025.xlsx', 'BP_Annual Permit Report-2018-2022.xlsx']


## Standing rules an agent reads first

An **agent instruction file** (commonly `CLAUDE.md` or `AGENTS.md`) is a markdown file the agent loads before doing anything — your project's *canonical source* of how to behave: where the real data lives, and the **non-negotiable rules** it must never break. This project has a real one. Let's read its opening:

In [4]:
from pathlib import Path
_f = REPO_ROOT/'CLAUDE.md'
_text = _f.read_text() if _f.exists() else None
if _text:
    print(_text[:1100])
else:
    # Colab fallback: a representative excerpt of this project's real CLAUDE.md
    print("# CLAUDE.md — Berkeley Housing Pipeline\n\nOrientation for any session working in `~/berkeley-data`. Verify specifics against\n`git log` and the DB before relying on them — this file is a map, not ground truth.\n\n## What this project is\nAn **independent** reconstruction of Berkeley's housing-production pipeline\n(entitlement → building permit → certificate of occupancy), built from **primary\nsources**, used to produce/verify the HCD Annual Progress Report (APR) and a\npublic explorer (berkeleybuild.com) + Datasette.\n\n## Canonical database\n**`databases/berkeley_housing_v2.db`** — V2 normalized schema (46 tables):\n`projects → project_versions → unit_program(+affordability) → project_parcels →\nparcels`, plus `project_events` (timeline: entitlement/BP/CO milestones via\n`vocabulary_event_types`), `permits`, `project_classifications`. The flat\ncompatibility view is **`v_projects_flat`** (what `generate_apr_v2.py` and\n`export_explorer_data_v2.py` read).\n- **V1 `berkeley_housing_analysis.db`** (flat")

# CLAUDE.md — Berkeley Housing Pipeline

Orientation for any session working in `~/berkeley-data`. Verify specifics against
`git log` and the DB before relying on them — this file is a map, not ground truth.

## What this project is
An **independent** reconstruction of Berkeley's housing-production pipeline
(entitlement → building permit → certificate of occupancy), built from **primary
sources**, used to produce/verify the HCD Annual Progress Report (APR) and a
public explorer (berkeleybuild.com) + Datasette.

## Canonical database
**`databases/berkeley_housing_v2.db`** — V2 normalized schema (46 tables):
`projects → project_versions → unit_program(+affordability) → project_parcels →
parcels`, plus `project_events` (timeline: entitlement/BP/CO milestones via
`vocabulary_event_types`), `permits`, `project_classifications`. The flat
compatibility view is **`v_projects_flat`** (what `generate_apr_v2.py` and
`export_explorer_data_v2.py` read).
- **V1 `berkeley_housing_analysis.db`** (flat

## The rules, pulled out

The file isn't prose to admire — it's *structured rules an agent obeys*. Pull out the lines that set the hardest constraints (the ones this whole project ran under): snapshot before writing, treat the canonical data as read-only, never push without permission.

In [5]:
import re
lines = (_text or '').splitlines()
# keep only the lines that state the hardest rules (snapshot, read-only, never push, verification target)
hits = [ln.strip(' #*-') for ln in lines
        if re.search(r'snapshot|read-only|never (commit|push)|verification target', ln, re.I)]
if not hits:
    # Colab fallback: the same rules, in case the file wasn't read
    hits = ['Snapshot before any DB write.', 'CKAN/HCD is the verification target, never a data source.',
            'Never commit/push without instruction.', 'Read-only by default.']
for h in hits[:6]:
    print('•', h[:110])

• VERIFICATION TARGET, never a data source** (see rules below).
• There are ~40 DB files total; most are dated snapshots/backups. Inventory:
• 1. **CKAN/HCD is the verification target, never a data source.** Build only from
• 2. **Snapshot before any DB write.** `cp` to `databases/keep_snapshot_<date>_pre-<change>.db`,
• confirm size + `PRAGMA integrity_check`. Then a **read-only preview → STOP for
• 3. **Read-only by default.** No merge/archive/delete/ingest/schema-change without


## Anatomy of a good one

Notice the shape of what you just read: it says **what the project is**, **where the canonical data lives**, the **rules that must never break**, and the **working discipline**. That's the whole recipe. To write your own, start tiny: one paragraph on what it is, one line on where the data lives, a short list of what to never do.

## Closing the on-ramp

You now hold the whole vocabulary: **open data → notebook → function → DataFrame → charts → tools → agents → instruction file.** You've seen the city's permits as a wall of cells, as honest numbers, as several pictures, as a 3D map — and you've checked a real number against the city's own report.

**Next — JN1 · Getting the data in the door.** The course proper begins: we turn the raw permit feed into a table you can trust, and meet the first way raw data tries to fool you.